# MIP Logistic Regression to sklearn

Use the high-level `FederatedLogisticRegression` class with a transient experiment JSON payload.
It runs only `logistic_regression` through platform-backend and returns a ready-to-use sklearn model.

In [ ]:
import numpy as np
import pandas as pd
import joblib
from platform_backend_client import configure, FederatedLogisticRegression

configure()


In [ ]:
# Fill this JSON payload with your transient experiment input.
# Only logistic_regression is supported by FederatedLogisticRegression.
payload = {
    "name": "Notebook logistic regression",
    "data_model": "dementia:0.1",
    "datasets": ["edsd"],
    "x": ["age_value", "education_level"],
    "y": ["gender"],
    "parameters": {"positive_class": "M"},
    "filters": None,
    "preprocessing": {},
    "mip_version": "9.0.0",
}
payload


In [ ]:
model = FederatedLogisticRegression(payload)


print("experiment uuid:", model.experiment_uuid)
print("status:", model.status)
print("classes_:", model.classes_)
print("coef_.shape:", model.coef_.shape)
print("intercept_:", model.intercept_)


In [ ]:
x_new = np.zeros((3, model.n_features_in_), dtype=float)
if model.n_features_in_ >= 1:
    x_new[1, 0] = 1.0
if model.n_features_in_ >= 2:
    x_new[2, 1] = 1.0

feature_names = getattr(model, "feature_names_in_", [f"x{i}" for i in range(model.n_features_in_)])
x_new_df = pd.DataFrame(x_new, columns=list(feature_names))

print("predict:", model.predict(x_new_df))
print("predict_proba:")
model.predict_proba(x_new_df)


In [ ]:
model_path = "logistic_regression_from_platform.joblib"
model.dump(model_path)
reloaded_model = joblib.load(model_path)
print("saved and reloaded model from:", model_path)
print("reloaded coef_.shape:", reloaded_model.coef_.shape)


You can also initialize from a JSON string with `FederatedLogisticRegression.from_json(json_string)`.